<a href="https://colab.research.google.com/github/freitas-econophys/Fisica_com_python1/blob/main/ridge_plot_Cv_liquidos/TernaryPlot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Tarefa 1:

#Disciplina: Física com Python 1 (2026.2) — CBPF
#Autor: [Nome do Aluno]
#Obs: Código desenvolvido para fins de avaliação acadêmica. Disponibilizado publicamente sob a Licença MIT para consulta educacional.

# Q2) Ternary plots.

In [3]:
# Definindo a função de gap de energia:
import numpy as np

def delta_E(sigma, alpha, k, V=1, TD=1):
    sigma_D = sigma / (2*np.pi**2*alpha)**(1/3)

    term1 = (2 + k*sigma_D**2) * np.sqrt(1 + k*sigma_D**2)

    term2 = k**2 * sigma**4 * np.log(
        (1 + np.sqrt(1 + k*sigma_D**2)) / (np.sqrt(k)*sigma)
    )

    return V*TD**4/(8*np.pi**2) * (term1 - term2 - 1)

In [5]:
# Cálculo das sensibilidades:

def sensitivities(sigma, alpha, k, eps=1e-5):

    f = delta_E(sigma, alpha, k)

    f_sigma_plus = delta_E(sigma*(1 + eps), alpha, k)
    f_alpha_plus = delta_E(sigma, alpha*(1 + eps), k)
    f_k_plus = delta_E(sigma, alpha, k*(1 + eps))

    S_sigma = abs((f_sigma_plus - f) / (eps*f))
    S_alpha = abs((f_alpha_plus - f) / (eps*f))
    S_k = abs((f_k_plus - f) / (eps*f))

    return S_sigma, S_alpha, S_k

#Testando em um ponto:

S = sensitivities(0.5, 1.0, 2)

print(S)

(np.float64(0.6133397114007288), np.float64(0.10311710444881636), np.float64(0.306667367261612))


In [6]:
# Normalização das sensibilidades:

def normalized_sensitivities(sigma, alpha, k, eps=1e-5):

    S_sigma, S_alpha, S_k = sensitivities(sigma, alpha, k, eps)

    S_total = S_sigma + S_alpha + S_k

    w_sigma = S_sigma / S_total
    w_alpha = S_alpha / S_total
    w_k = S_k / S_total

    return w_sigma, w_alpha, w_k

#Teste
w_sigma, w_alpha, w_k = normalized_sensitivities(0.5, 1.0, 2)

print("w_sigma =", w_sigma)
print("w_alpha =", w_alpha)
print("w_k     =", w_k)

print("Soma =", w_sigma + w_alpha + w_k)



w_sigma = 0.5994772888034576
w_alpha = 0.10078649899101565
w_k     = 0.2997362122055267
Soma = 1.0


In [7]:
#Ternary plot das sensibilidades

import numpy as np
import pandas as pd
import plotly.express as px

# Intervalos dos parâmetros
sigma_values = np.linspace(0.1, 1.0, 20)
alpha_values = np.linspace(0.1, 1.0, 20)
k_values = np.linspace(1, 10, 20)

# Lista para armazenar os resultados
data = []

for sigma in sigma_values:
    for alpha in alpha_values:
        for k in k_values:

            w_sigma, w_alpha, w_k = normalized_sensitivities(
                sigma, alpha, k
            )

            data.append({
                "sigma": w_sigma,
                "alpha": w_alpha,
                "k": w_k
            })

# DataFrame
df = pd.DataFrame(data)

# Ternary plot
fig = px.scatter_ternary(
    df,
    a="sigma",
    b="alpha",
    c="k"
)

fig.update_layout(
    ternary={
        "aaxis": {"title": "σ"},
        "baxis": {"title": "α"},
        "caxis": {"title": "k"}
    }
)

fig.show()


In [9]:

#Melhorando a resolução
import numpy as np
import pandas as pd
import plotly.express as px


def delta_E(sigma, alpha, k, V=1, TD=1):

    sigma_D = sigma / (2*np.pi**2*alpha)**(1/3)

    term1 = (2 + k*sigma_D**2) * np.sqrt(1 + k*sigma_D**2)

    term2 = k**2 * sigma**4 * np.log(
        (1 + np.sqrt(1 + k*sigma_D**2)) /
        (np.sqrt(k)*sigma)
    )

    return V*TD**4/(8*np.pi**2) * (term1 - term2 - 1)


def sensitivities(sigma, alpha, k, eps=1e-5):

    f = delta_E(sigma, alpha, k)

    f_sigma_plus = delta_E(sigma*(1 + eps), alpha, k)
    f_alpha_plus = delta_E(sigma, alpha*(1 + eps), k)
    f_k_plus = delta_E(sigma, alpha, k*(1 + eps))

    S_sigma = abs((f_sigma_plus - f) / (eps*f))
    S_alpha = abs((f_alpha_plus - f) / (eps*f))
    S_k = abs((f_k_plus - f) / (eps*f))

    return S_sigma, S_alpha, S_k


def normalized_sensitivities(sigma, alpha, k, eps=1e-5):

    S_sigma, S_alpha, S_k = sensitivities(sigma, alpha, k, eps)

    S_total = S_sigma + S_alpha + S_k

    w_sigma = S_sigma / S_total
    w_alpha = S_alpha / S_total
    w_k = S_k / S_total

    return w_sigma, w_alpha, w_k


sigma_values = np.linspace(0.1, 1.0, 5)
alpha_values = np.linspace(0.1, 1.0, 5)
k_values = np.linspace(1, 5, 5)

#Gerando os dadso:

data = []

for sigma in sigma_values:
    for alpha in alpha_values:
        for k in k_values:

            try:

                w_sigma, w_alpha, w_k = normalized_sensitivities(
                    sigma, alpha, k
                )

                data.append([
                    sigma,
                    alpha,
                    k,
                    w_sigma,
                    w_alpha,
                    w_k
                ])

            except (ValueError, ZeroDivisionError, FloatingPointError):

                pass


df = pd.DataFrame(
    data,
    columns=[
        "sigma",
        "alpha",
        "k",
        "w_sigma",
        "w_alpha",
        "w_k"
    ]
)


print(df.head())


fig = px.scatter_ternary(
    df,
    a="w_sigma",
    b="w_alpha",
    c="w_k",
    hover_data=["sigma", "alpha", "k"]
)

fig.update_traces(mode="markers")

fig.update_layout(
    title="Sensibilidade relativa de ΔEk"
)

fig.show()

   sigma  alpha    k   w_sigma   w_alpha       w_k
0    0.1    0.1  1.0  0.541019  0.188472  0.270508
1    0.1    0.1  2.0  0.537491  0.193765  0.268744
2    0.1    0.1  3.0  0.534251  0.198625  0.267124
3    0.1    0.1  4.0  0.531178  0.203234  0.265588
4    0.1    0.1  5.0  0.528219  0.207673  0.264108
